In [5]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

REPO_ROOT = Path.cwd() if (Path.cwd() / "analysis").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
from analysis import acs

df = acs.load_level("tract")
df.head(5)

,STATE,COUNTY,TRACT,NAME,B01003_001E,B01003_001M,B19013_001E,B19013_001M,B17001_002E,B17001_002M,B01001B_014E,B01001B_014M,B01001B_015E,B01001B_015M,B01001B_016E,B01001B_016M,B01001B_029E,B01001B_029M,B01001B_030E,B01001B_030M,B01001B_031E,B01001B_031M
0,34,001,000100,Census Tract 1; Atlantic County; New Jersey,2380,475,56222.0,13213.0,478,297,1,3,0,14,0,14,0,14,0,14,0,14
1,34,001,000200,Census Tract 2; Atlantic County; New Jersey,3009,549,42346.0,28560.0,624,357,0,14,0,14,0,14,0,14,0,14,0,14
2,34,001,000300,Census Tract 3; Atlantic County; New Jersey,4064,607,52763.0,18124.0,1149,509,0,14,0,14,0,14,0,14,0,14,0,14
3,34,001,000400,Census Tract 4; Atlantic County; New Jersey,3496,493,50104.0,13921.0,985,451,18,27,19,29,0,14,0,14,0,14,0,14
4,34,001,000500,Census Tract 5; Atlantic County; New Jersey,2944,555,53378.0,14617.0,732,281,0,14,0,14,0,14,0,14,0,14,0,14


In [6]:
GEO_COLUMNS = {
    "STATE": "State FIPS code",
    "COUNTY": "County FIPS code (within state)",
    "TRACT": "Census tract code (within county)",
    "BLOCK_GROUP": "Block group code (within tract)",
    "NAME": "Human-readable geography label from the Census API",
}


def column_description(col: str) -> str:
    if col in GEO_COLUMNS:
        return GEO_COLUMNS[col]
    if col.endswith("E"):
        base = col[:-1]
        if base in acs.VARIABLES:
            return f"{acs.VARIABLES[base]} (estimate)"
    if col.endswith("M"):
        base = col[:-1]
        if base in acs.VARIABLES:
            return f"{acs.VARIABLES[base]} (margin of error, 90% confidence)"
    return "Unknown column"


column_map = pd.DataFrame({
    "column": df.columns,
    "description": [column_description(c) for c in df.columns],
})

pd.set_option("display.max_colwidth", None)
column_map

,column,description
0,STATE,State FIPS code
1,COUNTY,County FIPS code (within state)
2,TRACT,Census tract code (within county)
3,NAME,Human-readable geography label from the Census API
4,B01003_001E,Total population (estimate)
5,B01003_001M,"Total population (margin of error, 90% confidence)"
6,B19013_001E,Median household income (estimate)
7,B19013_001M,"Median household income (margin of error, 90% confidence)"
8,B17001_002E,People below poverty level (estimate)
9,B17001_002M,"People below poverty level (margin of error, 90% confidence)"


In [7]:
from analysis import decennial

df_dhc = decennial.load_dhc("block")
df_dhc.head(5)

,STATE,COUNTY,TRACT,BLOCK,NAME,P1_001N,P12B_020N,P12B_021N,P12B_022N,P12B_023N,P12B_024N,P12B_025N,P12B_044N,P12B_045N,P12B_046N,P12B_047N,P12B_048N,P12B_049N
0,34,001,010501,4106,"Block 4106, Block Group 4, Census Tract 105.01, Atlantic County, New Jersey",0,0,0,0,0,0,0,0,0,0,0,0,0
1,34,001,010501,4103,"Block 4103, Block Group 4, Census Tract 105.01, Atlantic County, New Jersey",0,0,0,0,0,0,0,0,0,0,0,0,0
2,34,001,010501,4104,"Block 4104, Block Group 4, Census Tract 105.01, Atlantic County, New Jersey",0,0,0,0,0,0,0,0,0,0,0,0,0
3,34,001,010501,4105,"Block 4105, Block Group 4, Census Tract 105.01, Atlantic County, New Jersey",0,0,0,0,0,0,0,0,0,0,0,0,0
4,34,001,010501,4107,"Block 4107, Block Group 4, Census Tract 105.01, Atlantic County, New Jersey",0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
DHC_GEO_COLUMNS = {
    "STATE": "State FIPS code",
    "COUNTY": "County FIPS code (within state)",
    "TRACT": "Census tract code (within county)",
    "BLOCK_GROUP": "Block group code (within tract)",
    "BLOCK": "Census block code (within block group)",
    "NAME": "Human-readable geography label from the Census API",
}

# Same labels as ingestion/pull_dhc_nj.py
DHC_VARIABLES = {
    "P1_001N": "Total population (table P1)",
    "P12B_020N": "Black male 65-66",
    "P12B_021N": "Black male 67-69",
    "P12B_022N": "Black male 70-74",
    "P12B_023N": "Black male 75-79",
    "P12B_024N": "Black male 80-84",
    "P12B_025N": "Black male 85+",
    "P12B_044N": "Black female 65-66",
    "P12B_045N": "Black female 67-69",
    "P12B_046N": "Black female 70-74",
    "P12B_047N": "Black female 75-79",
    "P12B_048N": "Black female 80-84",
    "P12B_049N": "Black female 85+",
}


def dhc_column_description(col: str) -> str:
    if col in DHC_GEO_COLUMNS:
        return DHC_GEO_COLUMNS[col]
    if col in DHC_VARIABLES:
        return f"{DHC_VARIABLES[col]} (full count — DHC publishes no MOE)"
    return "Unknown column"


column_map_dhc = pd.DataFrame({
    "column": df_dhc.columns,
    "description": [dhc_column_description(c) for c in df_dhc.columns],
})

pd.set_option("display.max_colwidth", None)
column_map_dhc

,column,description
0,STATE,State FIPS code
1,COUNTY,County FIPS code (within state)
2,TRACT,Census tract code (within county)
3,BLOCK,Census block code (within block group)
4,NAME,Human-readable geography label from the Census API
5,P1_001N,Total population (table P1) (full count — DHC publishes no MOE)
6,P12B_020N,Black male 65-66 (full count — DHC publishes no MOE)
7,P12B_021N,Black male 67-69 (full count — DHC publishes no MOE)
8,P12B_022N,Black male 70-74 (full count — DHC publishes no MOE)
9,P12B_023N,Black male 75-79 (full count — DHC publishes no MOE)
